Partition and move criteo raw data to s3

In [ ]:
from datasets import load_dataset
import os

full = load_dataset(
    'csv',
    data_files="dataset/criteo_attribution_dataset.tsv",
    delimiter='\t'
)
full

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
import joblib

# list features that will require embeddings and print their cardinality
sparse_features = ["uid", "campaign"]+[f"cat{i}" for i in range(1, 10)]
sparse_voc_size = list()
V = dict()
for f in sparse_features:
    unique_values = full["train"].unique(f)
    
    
    print(f, len(unique_values))
    sparse_voc_size.append(len(unique_values))
    V[f] = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=len(unique_values)+1, dtype=int)
    V[f].fit([[val] for val in unique_values])

joblib.dump(V, "assets/ray/ordinal_encoders.joblib", compress=3)
print(sparse_features)


In [18]:
V = joblib.load("assets/ray/ordinal_encoders.joblib")

uid_encoder = V["uid"]
encoded = uid_encoder.transform([[26436149], [18382979], [2159461], [5675063], [17610544], [10809250]])
print(encoded)

[[5002913]
 [3478401]
 [ 409335]
 [1074072]
 [3332168]
 [2045483]]


In [ ]:
import s3fs
import polars as pl
from datetime import datetime, timezone

# import math

S3_BUCKET = "<MY-BUCKET>"
S3_PREFIX = "criteo/unprocessed"
# N_SHARDS = 20

fs = s3fs.S3FileSystem()

dataset = full["train"]
# shard_size = math.ceil(len(dataset) / N_SHARDS)

# for i in range(N_SHARDS):
#     start = i * shard_size
#     end = min(start + shard_size, len(dataset))
#     shard = dataset.select(range(start, end))
#     s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/part-{i:05d}.parquet"
#     with fs.open(s3_path, "wb") as f:
#         shard.to_parquet(f)
#     print(f"[{i+1}/{N_SHARDS}] written {len(shard)} rows → {s3_path}")

now = datetime.now(timezone.utc).replace(tzinfo=None)
# fast vectorized grouping
df = dataset.to_polars()

# create timestamp to represent an actual timestamp, criteo default timestamp is a relative number
df = df.with_columns(
    (pl.lit(now) + pl.duration(seconds=pl.col("timestamp") - df["timestamp"].max()))
    # .dt.strftime("%Y-%m-%d %H:%M:%S")
    .dt.replace_time_zone(None)
    .cast(pl.Datetime("us"))
    .alias("event_timestamp")
)

# Add year, month, day columns derived from the timestamp
df = df.with_columns([
    pl.col("event_timestamp").dt.year().alias("year"),
    pl.col("event_timestamp").dt.month().alias("month"),
    pl.col("event_timestamp").dt.day().alias("day"),
])

# Partition by year, month, day
partitions = df.partition_by(["year", "month", "day"], as_dict=True)

for keys, part in partitions.items():
    year, month, day = keys
    s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/year={year}/month={month:02d}/day={day:02d}/data.parquet"
    with fs.open(s3_path, "wb") as f:
        part.drop(["year", "month", "day"]).write_parquet(
            f,
            use_pyarrow=True,
            pyarrow_options={
                "coerce_timestamps": "us",
                "allow_truncated_timestamps": True,
            }
        )
    print(f"year={year} month={month} day={day}: {len(part)} rows → {s3_path}")


In [32]:
df.schema

Schema([('timestamp', Int64),
        ('uid', Int64),
        ('campaign', Int64),
        ('conversion', Int64),
        ('conversion_timestamp', Int64),
        ('conversion_id', Int64),
        ('attribution', Int64),
        ('click', Int64),
        ('click_pos', Int64),
        ('click_nb', Int64),
        ('cost', Float64),
        ('cpo', Float64),
        ('time_since_last_click', Int64),
        ('cat1', Int64),
        ('cat2', Int64),
        ('cat3', Int64),
        ('cat4', Int64),
        ('cat5', Int64),
        ('cat6', Int64),
        ('cat7', Int64),
        ('cat8', Int64),
        ('cat9', Int64),
        ('event_timestamp', Datetime(time_unit='us', time_zone=None)),
        ('year', Int32),
        ('month', Int8),
        ('day', Int8)])

In [ ]:
from datasets import Dataset

data_path = "<MY-BUCKET>/criteo/unprocessed/year=2026/month=03/day=20/*"
raw_df = Dataset.from_parquet(f"s3://{data_path}")
raw_pl = raw_df.to_polars()

Generating train split: 552290 examples [00:00, 3320426.29 examples/s]


In [ ]:
raw_pl.head(10)

timestamp,uid,campaign,conversion,conversion_timestamp,conversion_id,attribution,click,click_pos,click_nb,cost,cpo,time_since_last_click,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,event_timestamp
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,datetime[μs]
2194007,26436149,32452111,0,-1,-1,0,1,-1,-1,0.001041,0.114012,47005,25259032,9312274,8181301,29196072,5824237,1973606,6288950,9312274,9491354,2026-03-20 00:00:00.084127
2194007,18382979,8904991,0,-1,-1,0,0,-1,-1,0.000024,0.018243,-1,25259032,9312274,18535518,29196072,11409686,1973606,9312274,3225256,29196072,2026-03-20 00:00:00.084127
2194007,2159461,27385784,0,-1,-1,0,0,-1,-1,0.00001,0.198589,-1,30763035,9312274,9050997,29196072,32440044,1973606,9312274,9312274,29196072,2026-03-20 00:00:00.084127
2194007,5675063,4800911,0,-1,-1,0,1,-1,-1,0.000029,0.142372,-1,30763035,9312274,9640264,29196072,5824237,29196072,9312274,29196072,29196072,2026-03-20 00:00:00.084127
2194007,17610544,5044698,0,-1,-1,0,1,-1,-1,0.000147,0.222113,-1,9312274,29196072,23631867,29196072,32440047,29196072,9312274,29196072,29196072,2026-03-20 00:00:00.084127
2194007,10809250,5281156,0,-1,-1,0,0,-1,-1,0.00011,0.322338,-1,30763035,9312274,28607196,29196072,5824239,29196072,9312274,29196072,29196072,2026-03-20 00:00:00.084127
2194008,30948132,7686710,0,-1,-1,0,0,-1,-1,0.00001,0.234656,-1,30763035,9312274,1248313,29196072,5824239,29196072,10003874,29196072,29196072,2026-03-20 00:00:01.084127
2194008,28714471,13365547,1,2737772,12832176,1,1,3,11,0.000761,0.030123,21,28928366,26597094,31616034,29196072,11409684,28928366,14426956,29196072,358249,2026-03-20 00:00:01.084127
2194008,14430528,15398570,0,-1,-1,0,1,-1,-1,0.000106,0.232128,-1,30763035,138937,4506847,29196072,32440044,1973606,19031884,32440044,29196072,2026-03-20 00:00:01.084127


View output of ray job
- categorigal data transformed using mappings from sckiti-learn OrdinalEncoder
- sequential window aggregation for last_5_clicks and last_5_conversions

In [ ]:
from datasets import load_dataset

data_path = "<MY-BUCKET>/feast/criteo/transformed/features/year=2026/month=03/day=20/*"

# Use force_redownload to ignore cache
transformed_df = load_dataset(
    "parquet", 
    data_files=f"s3://{data_path}", 
    download_mode="force_redownload",
)


Generating train split: 552290 examples [00:00, 5645003.04 examples/s]


In [10]:
transformed_df["train"].to_polars().head()

event_timestamp,uid,campaign,click,conversion,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,last_5_clicks,last_5_conversions
datetime[μs],f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
2026-03-20 07:04:02.084127,1760.0,198.0,0,0,5.0,9.0,82.0,17.0,39.0,1.0,41017.0,4.0,25.0,"""""",""""""
2026-03-20 04:12:29.084127,4074.0,294.0,0,0,8.0,23.0,190.0,17.0,47.0,25.0,21626.0,8.0,17.0,"""""",""""""
2026-03-20 07:07:59.084127,10433.0,49.0,1,0,0.0,23.0,967.0,17.0,14.0,1.0,5736.0,0.0,25.0,"""""",""""""
2026-03-20 01:00:19.084127,12521.0,501.0,1,0,7.0,23.0,271.0,17.0,25.0,23.0,10741.0,8.0,25.0,"""""",""""""
2026-03-20 10:36:21.084127,12521.0,501.0,0,0,7.0,23.0,1293.0,17.0,25.0,23.0,16276.0,8.0,8.0,"""501.0""",""""""
